In [2]:
export MODEL="unsloth/Qwen3.6-27B-GGUF:IQ4_NL"

In [3]:
llama-fit-params -hf $MODEL # This is just for downloading and some initial values to probe

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 5737 MiB):
  Device 0: NVIDIA GeForce RTX 2060, compute capability 7.5, VMM: yes, VRAM: 5737 MiB
load_backend: loaded CUDA backend from /app/libggml-cuda.so
load_backend: loaded CPU backend from /app/libggml-cpu-haswell.so
common_download_file_single_online: HEAD failed, status: 404
no remote preset found, skipping

common_params_fit_impl: getting device memory data for initial parameters:[K
common_memory_breakdown_print: | memory breakdown [MiB] | total   free     self   model   context   compute       unaccounted |
common_memory_breakdown_print: |   - CUDA0 (RTX 2060)   |  5737 = 5191 + (31992 = 14634 +   16533 +     824) + 17592186012969 |
common_memory_breakdown_print: |   - Host               |                  1214 =   682 +       0 +     532                   |
common_params_fit_impl: projected to use 31992 MiB of device memory vs. 5191 MiB of free device memory
common_params_fit_impl: cannot meet free memory target of 1024 MiB, 

In [4]:
llama-fit-params -hf $MODEL # This is just for downloading and some initial values to probe

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 5737 MiB):
  Device 0: NVIDIA GeForce RTX 2060, compute capability 7.5, VMM: yes, VRAM: 5737 MiB
load_backend: loaded CUDA backend from /app/libggml-cuda.so
load_backend: loaded CPU backend from /app/libggml-cpu-haswell.so
common_download_file_single_online: HEAD failed, status: 404
no remote preset found, skipping
common_params_fit_impl: getting device memory data for initial parameters:
common_memory_breakdown_print: | memory breakdown [MiB] | total   free     self   model   context   compute       unaccounted |
common_memory_breakdown_print: |   - CUDA0 (RTX 2060)   |  5737 = 4931 + (31992 = 14634 +   16533 +     824) + 17592186013229 |
common_memory_breakdown_print: |   - Host               |                  1214 =   682 +       0 +     532                   |
common_params_fit_impl: projected to use 31992 MiB of device memory vs. 4931 MiB of free device memory
common_params_fit_impl: cannot meet free memory target of 1024 MiB, nee

In [3]:
# Configurable batch sizes to test to fit context
# Though it looks like batch sizes have no effects 
# as per README 
# "Increasing this value above the value of the physical batch size may improve prompt processing performance 
# when using multiple GPUs with pipeline parallelism. Default: `2048`" 

BATCH_SIZES="2048" # 2 args so we can see the diff, shoud not make a difference  
UBATCH_SIZES="512 1024 1440 2048"
FITT="512"
CMOE="99"

echo "Testing different batch/ubatch/fitt combinations for ${MODEL}..." 
  
for ub in $UBATCH_SIZES; do  
    for b in $BATCH_SIZES; do
        for ft in $FITT; do
            for cmoe in $CMOE; do
                # need to add fitt for nvidia
                OUTPUT=$(llama-fit-params -hf $MODEL -b $b -ub $ub -fitt $ft -ncmoe $cmoe 2>&1)
                
                #echo "Raw output: $OUTPUT"  # Debug line  
                # Extract context and GPU layers more robustly  
                CTX=$(echo "$OUTPUT" | grep -o '\-c [0-9]*' | awk '{print $2}')  
                NGL=$(echo "$OUTPUT" | grep -o '\-ngl -\?[0-9]*' | awk '{print $2}')  
                
                if [ ! -z "$CTX" ] && [ ! -z "$NGL" ]; then  
                    echo "ub: $ub, b: $b, fitt: $ft, ncmoe: $cmoe, ctx: $CTX, ngl: $NGL"  
                else  
                    echo "No valid parameters found"  
                fi
            done
        done
    done  
done 

Testing different batch/ubatch/fitt combinations for unsloth/Qwen3.6-35B-A3B-GGUF:UD-IQ4_XS...
ub: 512, b: 2048, fitt: 512, ncmoe: 99, ctx: 121856, ngl: -1
ub: 1024, b: 2048, fitt: 512, ncmoe: 99, ctx: 93696, ngl: -1
ub: 1440, b: 2048, fitt: 512, ncmoe: 99, ctx: 72960, ngl: -1
ub: 2048, b: 2048, fitt: 512, ncmoe: 99, ctx: 45312, ngl: -1


In [7]:
llama-bench -hf $MODEL -ngl 17 --mmap 0 -p 1000,2000 -n 256,512

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 5737 MiB):
  Device 0: NVIDIA GeForce RTX 2060, compute capability 7.5, VMM: yes, VRAM: 5737 MiB
load_backend: loaded CUDA backend from /app/libggml-cuda.so
load_backend: loaded CPU backend from /app/libggml-cpu-haswell.so
| model                          |       size |     params | backend    | ngl | mmap |            test |                  t/s |
| ------------------------------ | ---------: | ---------: | ---------- | --: | ---: | --------------: | -------------------: |
| qwen35 27B IQ4_NL - 4.5 bpw    |  14.96 GiB |    26.90 B | CUDA       |  17 |    0 |          pp1000 |        205.07 ± 3.32 |
| qwen35 27B IQ4_NL - 4.5 bpw    |  14.96 GiB |    26.90 B | CUDA       |  17 |    0 |          pp2000 |        202.43 ± 8.86 |
| qwen35 27B IQ4_NL - 4.5 bpw    |  14.96 GiB |    26.90 B | CUDA       |  17 |    0 |           tg256 |          1.47 ± 0.03 |
| qwen35 27B IQ4_NL - 4.5 bpw    |  14.96 GiB |    26.90 B | CUDA       |  17 |    0 | 

In [4]:
echo "Staring llama-bench for ${MODEL}..." 
llama-bench -hf $MODEL -fitt 512 -ub 512,1024 --mmap 0 -ngl 99 -ncmoe 99 -p 1000,2000 -n 256,512

Staring llama-bench for unsloth/Qwen3.6-35B-A3B-GGUF:UD-IQ4_XS...
ggml_cuda_init: found 1 CUDA devices (Total VRAM: 5737 MiB):
  Device 0: NVIDIA GeForce RTX 2060, compute capability 7.5, VMM: yes, VRAM: 5737 MiB
load_backend: loaded CUDA backend from /app/libggml-cuda.so
load_backend: loaded CPU backend from /app/libggml-cpu-haswell.so
| model                          |       size |     params | backend    | ngl |  n_cpu_moe | n_ubatch | mmap |       fitt |            test |                  t/s |
| ------------------------------ | ---------: | ---------: | ---------- | --: | ---------: | -------: | ---: | ---------: | --------------: | -------------------: |
| qwen35moe 35B.A3B IQ4_XS - 4.25 bpw |  16.50 GiB |    34.66 B | CUDA       |  99 |         99 |      512 |    0 |        512 |          pp1000 |        456.45 ± 3.19 |
| qwen35moe 35B.A3B IQ4_XS - 4.25 bpw |  16.50 GiB |    34.66 B | CUDA       |  99 |         99 |      512 |    0 |        512 |          pp2000 |        442.51 

In [ ]:
llama-fit-params -hf $MODEL -ncmoe 32

In [ ]:
echo "Staring llama-cli for ${MODEL}..." 
llama-cli -lv 3 -hf $MODEL -ub 64 -fitt 256 -ngl -1 --no-warmup --no-mmproj --reasoning off --single-turn --prompt "Who are you?"

In [ ]:
echo "Staring llama-server for ${MODEL}..." 
llama-server -hf $MODEL --threads-http 1 -ub 64 -fitt 256 -ngl -1 --no-warmup --no-mmproj --reasoning off -np 1